In [13]:
from pathlib import Path

import MDAnalysis as mda
import torch
import numpy as np
from torch_geometric.data import HeteroData
from MDAnalysis.lib.distances import capped_distance
from sklearn.preprocessing import LabelEncoder

In [14]:
struct_file_path = "/home/phillip/Goethe/Thesis/lipid-graph-nn/data/membrane_only/DOPC100/run.gro"
top_file_path = "/home/phillip/Goethe/Thesis/lipid-graph-nn/data/membrane_only/DOPC100/run/prun.tpr"

spatial_cutoff = 11.0
atom_select_str = "not (resname W or name NA or name CL)"
#atom_select_str = "name NA"

In [15]:
u = mda.Universe(top_file_path, struct_file_path)
lipids = u.select_atoms(atom_select_str)
set(lipids.moltypes)

{'DOPC'}

In [16]:
n_nodes = len(lipids)
n_nodes

4704

In [17]:
le = LabelEncoder()
bead_types = le.fit_transform(lipids.names)
node_x = torch.tensor(bead_types, dtype=torch.long).unsqueeze(1)

In [18]:
global_to_local = np.full(u.atoms.n_atoms, -1, dtype=int) # type: ignore
global_to_local[lipids.indices] = np.arange(n_nodes)
global_to_local

array([ 0,  1,  2, ..., -1, -1, -1], shape=(10126,))

In [19]:
global_bonds = lipids.bonds.to_indices()
global_bonds

array([[   0,    1],
       [   1,    2],
       [   2,    3],
       ...,
       [4700, 4701],
       [4701, 4702],
       [4702, 4703]], shape=(4312, 2), dtype=int32)

In [20]:
local_bonds = global_to_local[global_bonds]
local_bonds

array([[   0,    1],
       [   1,    2],
       [   2,    3],
       ...,
       [4700, 4701],
       [4701, 4702],
       [4702, 4703]], shape=(4312, 2))

In [21]:
local_bonds[:, 1] != -1 & (local_bonds[:, 1] != -1)

array([False,  True,  True, ...,  True,  True,  True], shape=(4312,))

In [22]:
mask = (local_bonds[:, 0] != -1) & (local_bonds[:, 1] != -1)
mask

array([ True,  True,  True, ...,  True,  True,  True], shape=(4312,))

In [23]:
valid_local_bonds = local_bonds[mask]
valid_local_bonds

array([[   0,    1],
       [   1,    2],
       [   2,    3],
       ...,
       [4700, 4701],
       [4701, 4702],
       [4702, 4703]], shape=(4312, 2))

In [27]:
bond_index = torch.tensor(valid_local_bonds.T, dtype=torch.long)
bond_index = torch.cat([bond_index, bond_index.flip(0)], dim=1)
bond_index.shape

torch.Size([2, 8624])

In [35]:
bond_set = set(map(tuple, valid_local_bonds))
reversed_bond_set = {(b, a) for a, b in bond_set}
all_bonded_pairs_set = bond_set.union(reversed_bond_set)

In [49]:
bond_index.shape[1]

8624

In [51]:
u.trajectory[0]

< Timestep 0 with unit cell dimensions [110. 110. 100.  90.  90.  90.] >

In [52]:
current_pos = torch.tensor(lipids.positions, dtype=torch.float)
current_pos

tensor([[ 67.5500,  12.3100,  69.5000],
        [ 67.3900,  12.5300,  66.5000],
        [ 67.3300,  11.9400,  63.5000],
        ...,
        [ 25.6000, 107.1600,  42.5000],
        [ 25.3000, 106.9300,  45.5000],
        [ 25.5900, 106.5900,  48.5000]])

In [54]:
pairs, dists = capped_distance(
            lipids.positions, 
            lipids.positions, 
            max_cutoff=spatial_cutoff, 
            box=u.dimensions,
            return_distances=True
        )
pairs, dists

(array([[   0, 1706],
        [   0, 1705],
        [   0, 1704],
        ...,
        [4703, 1655],
        [4703, 1651],
        [4703, 1650]], shape=(249054, 2)),
 array([10.45829926,  9.09029134,  8.43607169, ...,  9.7625644 ,
         9.02389011,  9.96999481], shape=(249054,)))

In [61]:
spatial_set = set(map(tuple, pairs))
spatial_set = {p for p in spatial_set if p[0] != p[1]}
pure_spatial_set = spatial_set - all_bonded_pairs_set
len(pure_spatial_set), len(spatial_set)

(235726, 244350)

In [65]:
if len(pure_spatial_set) > 0:
    spatial_edges = np.array(list(pure_spatial_set))
    spatial_index = torch.tensor(spatial_edges.T, dtype=torch.long)
    spatial_index = torch.cat([spatial_index, spatial_index.flip(0)], dim=1)
else:
    spatial_index = torch.empty((2, 0), dtype=torch.long)
spatial_index.shape

torch.Size([2, 471452])

In [66]:
data = HeteroData()

In [69]:
data["bead"].x = node_x
data["bead"].num_nodes = n_nodes
data["bead", "bonded", "bead"].edge_index = bond_index

In [70]:
data["bead"].pos = current_pos
data["bead", "spatial", "bead"].edge_index = spatial_index

In [73]:
data["bead", "spatial", "bead"].num_edges

471452